# Mamba — TaskTracker-only

Pipeline dung 470 TaskTracker sessions va ba split CSV co dinh cua nhom: train=333/191 groups, validation=68/37 groups, test=69/43 groups (15 label 1, 54 label 0). Notebook khong tao split moi. PasteTrace khong duoc su dung.

In [ ]:
# Cell 1: clone/pull dung source tu GitHub
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
BRANCH = 'ModelMambaV2'
BASE_DIR = Path('/kaggle/working/Fraud-Detection')

def clone_into_clean_directory():
    for number in range(100):
        candidate = BASE_DIR if number == 0 else Path(f'{BASE_DIR}-clean-{number}')
        if not candidate.exists():
            subprocess.run(
                ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(candidate)],
                check=True,
            )
            return candidate
    raise RuntimeError('Cannot find an empty directory for a clean clone.')

REPO_DIR = BASE_DIR
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    checkout = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', BRANCH],
        text=True, capture_output=True,
    )
    pull = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH],
        text=True, capture_output=True,
    ) if checkout.returncode == 0 else checkout
    if checkout.returncode != 0 or pull.returncode != 0:
        print('Existing clone cannot fast-forward; using a clean clone instead.')
        print((checkout.stderr + '\n' + pull.stderr).strip())
        REPO_DIR = clone_into_clean_directory()
elif REPO_DIR.exists():
    print(f'{REPO_DIR} is not a git repository; using a clean clone instead.')
    REPO_DIR = clone_into_clean_directory()
else:
    REPO_DIR = clone_into_clean_directory()

os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Working directory:', Path.cwd())
print('Branch:', BRANCH, '| commit:', commit)

In [ ]:
# Cell 2: cai bang chinh Python cua Kaggle kernel; khong dung -q de thay loi that.
import importlib, subprocess, sys, torch
print('Python:', sys.version)
print('CUDA:', torch.cuda.is_available(), '| torch:', torch.__version__, '| torch CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not enabled. In Kaggle: Settings -> Accelerator -> GPU.')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'setuptools>=68', 'wheel', 'packaging', 'ninja',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--no-build-isolation', '--no-cache-dir', '-r', 'requirements-mamba.txt',
], check=True)
importlib.invalidate_caches()
from mamba_ssm import Mamba
device = torch.device('cuda')
probe_model = Mamba(d_model=64).to(device)
probe_input = torch.randn(2, 32, 64, device=device)
with torch.no_grad():
    probe_output = probe_model(probe_input)
print('Mamba output:', probe_output.shape, '| device:', probe_output.device)
del probe_model, probe_input, probe_output
torch.cuda.empty_cache()

In [ ]:
!python -m src.data.build_sequences --min-events 1
!python -m src.data.make_splits  # validate only; never creates a split

In [ ]:
import pandas as pd
index = pd.read_csv('data/mamba/index.csv')
expected = {
    'train': {'sessions': 333, 'groups': 191},
    'validation': {'sessions': 68, 'groups': 37},
    'test': {'sessions': 69, 'groups': 43, 'risk_1': 15, 'normal_0': 54},
}
for name in ['train', 'validation', 'test']:
    frame = pd.read_csv(f'data/splits/tasktracker/{name}.csv').merge(
        index[['session_id', 'group_id', 'label']], on='session_id', validate='one_to_one'
    )
    actual = {'sessions': len(frame), 'groups': frame.group_id.nunique()}
    if name == 'test':
        actual.update({'risk_1': int((frame.label == 1).sum()), 'normal_0': int((frame.label == 0).sum())})
    print(name, actual)
    assert actual == expected[name], f'{name}: expected {expected[name]}, found {actual}'

## Train + validation
Scaler chi fit train. Checkpoint va threshold duoc chon bang validation. Test split khong duoc nap trong qua trinh nay.

In [ ]:
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 3e-4 --patience 10 \
    --batch-size 8 --max-len 1000 --seed 42

## Held-out TaskTracker test
Lenh nay tao `predictions.csv`, `metrics.json`, `method.json` va luu ban sao ba split CSV ma chuong trinh thuc su doc. Runtime tren Tesla T4 chi mang tinh mo ta.

In [ ]:
!python -m src.models.mamba_model test
import json, pandas as pd
display(pd.read_csv('results/mamba/predictions.csv').head())
with open('results/mamba/metrics.json', encoding='utf8') as f:
    metrics = json.load(f)
metrics

# Kiem tra 69 dong va thu tu session khop chinh xac test.csv.
pred = pd.read_csv('results/mamba/predictions.csv')
test_ids = pd.read_csv('results/mamba/splits/test.csv').session_id.tolist()
assert len(pred) == 69 and pred.session_id.tolist() == test_ids
print('Verified: predictions.csv matches all 69 ordered test session IDs.')

In [ ]:
# Dong goi ket qua va dung ba split CSV da doc de tai ve.
from pathlib import Path
import zipfile
artifacts = [
    Path('results/mamba/predictions.csv'),
    Path('results/mamba/metrics.json'),
    Path('results/mamba/method.json'),
    Path('results/mamba/splits/train.csv'),
    Path('results/mamba/splits/validation.csv'),
    Path('results/mamba/splits/test.csv'),
    Path('results/mamba/training_history.csv'),
    Path('results/mamba/validation_predictions.csv'),
    Path('models/mamba/config.json'),
    Path('models/mamba/scaler.json'),
    Path('models/mamba/mamba.pt'),
]
missing = [str(path) for path in artifacts if not path.is_file()]
assert not missing, f'Missing artifacts: {missing}'
zip_path = Path('/kaggle/working/mamba_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in artifacts:
        archive.write(path, path.as_posix())
print('Saved:', zip_path)